In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Skin Cancer detection using a pre trained model VGG19.

# Dataset can be downloaded from
# https://www.kaggle.com/fanconic/skin-cancer-malignant-vs-benign

from keras.layers import Input, Lambda, Dense, Flatten
from keras.models import Model
#from keras.applications.vgg16 import VGG16
from keras.applications.vgg19 import VGG19
from keras.applications.vgg19 import preprocess_input
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
import random 
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout


In [ ]:
# re-size all the images to standard size 224 x 224
IMAGE_SIZE = [224, 224]

# specifying path for train and test data folders
train_path = '../input/skin-cancer-malignant-vs-benign/train'
valid_path = '../input/skin-cancer-malignant-vs-benign/test'

## Dataset image Visualization

In [ ]:
#function to visualize image
def plot_image(file, directory=None, sub=False, aspect=None):
    path = directory + file
    
    img = plt.imread(path)
    
    plt.imshow(img, aspect=aspect)
#     plt.title(file)
    plt.xticks([])
    plt.yticks([])
    
    if sub:
        plt.show()

In [ ]:
def plot_img_dir(directory=train_path, count=5):
    selected_files = random.sample(os.listdir(directory), count)
    
    ncols = 5
    nrows = count//ncols if count%ncols==0 else count//ncols+1
    
    figsize=(20, ncols*nrows)

    ticksize = 14
    titlesize = ticksize + 8
    labelsize = ticksize + 5


    params = {'figure.figsize' : figsize,
              'axes.labelsize' : labelsize,
              'axes.titlesize' : titlesize,
              'xtick.labelsize': ticksize,
              'ytick.labelsize': ticksize}

    plt.rcParams.update(params)
    
    i=0
    
    for file in selected_files:        
        plt.subplot(nrows, ncols, i+1)
        path = directory + file
        plot_image(file, directory, aspect=None)

        i=i+1
    
    plt.tight_layout()
    plt.show()
    
def plot_img_dir_main(directory=train_path, count=5):
    labels = os.listdir(directory)
    for label in labels:
        print(label)
        plot_img_dir(directory=directory+"/"+label+"/", count=count)
        

In [ ]:
plot_img_dir_main(directory=train_path, count=5)

## Model Implementation (VGG 19) using Transfer Learning

In [ ]:
# added preprocessing layer to the front of VGG
vgg = VGG19(input_shape=IMAGE_SIZE + [3], weights='imagenet', include_top=False)

In [ ]:
# Freezing the existing layers 
for layer in vgg.layers:
  layer.trainable = False

In [ ]:
# fetching number of classes
folders = glob('../input/skin-cancer-malignant-vs-benign/train/*')

In [ ]:
folders


In [ ]:
len(folders)

We will then build the last fully-connected layer. I have just used the basic settings, but feel free to experiment with different values of dropout, and different Optimisers and activation functions.

## Addition of fully connected layer in the last layers

In [ ]:
# Flatten the output layer to 1 dimension
#x = layers.Flatten()(base_model.output)
x = Flatten()(vgg.output)

# Add a fully connected layer with 512 hidden units and ReLU activation
x = Dense(512, activation='relu')(x)

# Add a dropout rate of 0.5
x = Dropout(0.5)(x)

# Add a fully connected layer with 256 hidden units and ReLU activation
x = Dense(256, activation='relu')(x)

# Add a dropout rate of 0.5
x = Dropout(0.5)(x)

# Add a final sigmoid layer for classification
prediction = Dense(len(folders), activation='softmax')(x)

#model = tf.keras.models.Model(base_model.input, x)

#model.compile(optimizer = tf.keras.optimizers.RMSprop(lr=0.0001), loss = 'binary_crossentropy',metrics = ['acc'])

In [ ]:
# model object creation
model = Model(inputs=vgg.input, outputs=prediction)

## Model Summary

In [ ]:
# view model structure
model.summary()


In [ ]:
# setting model cost and optimization method to use
model.compile(
  loss='categorical_crossentropy',
  optimizer='adam',
  metrics=['accuracy']
)


## Implementing Data Augmentation

In [ ]:
# Image Data Generator to import the images from the dataset
from keras.preprocessing.image import ImageDataGenerator

Image Augmentation
Since we took up a much smaller dataset of images earlier, we can make up for it by augmenting this data and increasing our dataset size. If you are working with the original larger dataset, you can skip this step and move straight on to building the model.

In [ ]:
# Addition of data-augmentation parameters to ImageDataGenerator
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   width_shift_range = 0.2,
                                   height_shift_range = 0.2,
                                   rotation_range = 40,
                                   horizontal_flip = True)
 

# validation data should not be augmented
test_datagen = ImageDataGenerator(rescale = 1./255)

training_set = train_datagen.flow_from_directory('../input/skin-cancer-malignant-vs-benign/train',
                                                 target_size = (224, 224),
                                                 batch_size = 64,
                                                 class_mode = 'categorical')

test_set = test_datagen.flow_from_directory('../input/skin-cancer-malignant-vs-benign/test',
                                            target_size = (224, 224),
                                            batch_size = 64,
                                            class_mode = 'categorical')

In [ ]:
# number of  benign test data
DIR = '../input/skin-cancer-malignant-vs-benign/test/benign'
benign_test = ([name for name in os.listdir(DIR) if os.path.isfile(os.path.join(DIR, name))])
print("number of benign test data:" + str(len(benign_test)))

In [ ]:
# number of malignant test data
DIR = '../input/skin-cancer-malignant-vs-benign/test/malignant'
malignant_test = ([name for name in os.listdir(DIR) if os.path.isfile(os.path.join(DIR, name))])
print("number of malignant test data:" + str(len(malignant_test)))